# Introduction to LangChain and Tools

This notebook demonstrates how **LangChain** simplifies the development of LLM applications.

We will cover two main concepts:
1.  **The "Simple" Way**: How to send a basic request to an LLM with just a few lines of code (wrapping the complexity of the API).
2.  **Integrations & Agents**: How to give the LLM access to real-time data using tools (specifically, the OpenWeatherMap API).

In [11]:
# Install required packages
# Note: you may need to restart the kernel to use updated packages.
%pip install -U -q langchain langchain-community langchain-openai pyowm python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Setup & Configuration

We use `python-dotenv` to load our API keys from a `.env` file, ensuring our credentials remain secure.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
# Assuming .env is in the parent directory or root
load_dotenv("../.env")

# Verify keys are loaded
if os.getenv("OPENAI_API_KEY"):
    print("✅ OPENAI_API_KEY loaded.")
else:
    print("❌ OPENAI_API_KEY not found!")

if os.getenv("OPENWEATHERMAP_API_KEY"):
    print("✅ OPENWEATHERMAP_API_KEY loaded.")
else:
    print("❌ OPENWEATHERMAP_API_KEY not found!")

✅ OPENAI_API_KEY loaded.
✅ OPENWEATHERMAP_API_KEY loaded.


## 2. The Simple Way: Chat with an LLM

Using the raw OpenAI API requires handling HTTP requests, JSON payloads, and session management. LangChain abstracts this into a simple `ChatOpenAI` class.

In [13]:
from langchain_openai import ChatOpenAI

# Initialize the model
# We use gpt-4o-mini for a balance of speed and cost
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Send a simple message
response = llm.invoke("Explain what LangChain is in one sentence.")

print("Response:")
print(response.content)

Response:
LangChain is a framework designed to facilitate the development of applications that utilize large language models (LLMs) by providing tools for integrating various data sources, managing workflows, and enhancing model interactions.


## 3. Integrations: Using Tools (Weather API)

LLMs are trained on past data and don't know the current weather. To fix this, we can give them **Tools**.

We will use the `OpenWeatherMap` tool. LangChain has a built-in wrapper for this, so we don't even need to write the API request code ourselves!

In [14]:
# Import the tool loader
from langchain_community.agent_toolkits.load_tools import load_tools

# Load the 'openweathermap-api' tool
# This automatically uses the OPENWEATHERMAP_API_KEY from our environment
tools = load_tools(["openweathermap-api"], llm)

## 4. Creating an Agent

An **Agent** is the brain that decides *which* tool to use and *when*.

We will use `create_tool_calling_agent`, which leverages the latest OpenAI function calling capabilities for reliable tool usage.

In [20]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# 1. Define the prompt template for the agent
# We strictly instruct the agent to pass ONLY the city name to avoid "NotFoundError" from the API
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. You have access to weather tools. "
               "When using the weather tool, strictly pass only the city name (e.g., 'London' or 'San Francisco'). "
               "Do not include phrases like 'right now', 'today', or 'in' as part of the location argument."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

# 2. Create the agent
agent = create_tool_calling_agent(llm, tools, prompt)

# 3. Create the executor (the runtime that actually runs the agent)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

### Let's test it!

We ask about the weather in a specific city. The agent should:
1.  Recognize it needs weather info.
2.  Call the `open_weather_map` tool.
3.  Read the API result.
4.  Answer our question.

In [17]:
response = agent_executor.invoke({"input": "What is the weather in San Francisco right now?"})

print("\nFinal Answer:")
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `open_weather_map` with `{'location': 'San Francisco'}`


In San Francisco, the current weather is as follows:
Detailed status: overcast clouds
Wind speed: 2.57 m/s, direction: 110°
Humidity: 91%
Temperature: 
  - Current: 7.28°C
  - High: 7.99°C
  - Low: 5.79°C
  - Feels like: 5.55°C
Rain: {}
Heat index: None
Cloud cover: 100%The current weather in San Francisco is overcast with a temperature of 7.28°C. The high for today is 7.99°C and the low is 5.79°C. It feels like 5.55°C. The humidity is at 91%, and there is a wind speed of 2.57 m/s coming from the east-southeast (110°). The cloud cover is 100%.

> Finished chain.

Final Answer:
The current weather in San Francisco is overcast with a temperature of 7.28°C. The high for today is 7.99°C and the low is 5.79°C. It feels like 5.55°C. The humidity is at 91%, and there is a wind speed of 2.57 m/s coming from the east-southeast (110°). The cloud cover is 100%.


### Try another location
Because the tools are integrated, we can ask about any city supported by the API.

In [18]:
response = agent_executor.invoke({"input": "Is it raining in London?"})

print("\nFinal Answer:")
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `open_weather_map` with `{'location': 'London'}`


In London, the current weather is as follows:
Detailed status: broken clouds
Wind speed: 6.26 m/s, direction: 237°
Humidity: 87%
Temperature: 
  - Current: 13.74°C
  - High: 14.19°C
  - Low: 12.95°C
  - Feels like: 13.44°C
Rain: {}
Heat index: None
Cloud cover: 75%It is not raining in London currently. The weather condition is broken clouds with a humidity of 87%.

> Finished chain.

Final Answer:
It is not raining in London currently. The weather condition is broken clouds with a humidity of 87%.


### Debugging Integration

If you encounter a `NotFoundError`, it usually means the OpenWeatherMap API could not find the location string exactly as passed.
Run the cell below to verify direct access to the tool.

In [ ]:
# Direct test of the wrapper to debug location strings
from langchain_community.utilities import OpenWeatherMapAPIWrapper
weather = OpenWeatherMapAPIWrapper()

print("Testing 'San Francisco' directly:")
try:
    print(weather.run("San Francisco"))
except Exception as e:
    print(f"Error with 'San Francisco': {e}")

print("\nTesting 'San Francisco, US' directly:")
try:
    print(weather.run("San Francisco, US"))
except Exception as e:
    print(f"Error with 'San Francisco, US': {e}")
